# PySpark Fundamentos — Daniel Guzmán

**Dataset:** Financial Transactions (Caixabank Tech)  
**Fecha:** 2026-04-25  
**Objetivo:** Cargar, explorar y transformar el dataset de transacciones usando PySpark puro.

In [0]:
MI_NOMBRE = "daniel"

catalog = "workspace"
schema = "default"
volume = f"week_2_{MI_NOMBRE}"

path_volume = f"/Volumes/{catalog}/{schema}/{volume}"

spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalog}.{schema}.{volume}")

print("Volume creado o ya existente:")
print(path_volume)

In [0]:
MI_NOMBRE = "daniel"

catalog = "workspace"
schema = "default"
volume = f"week_2_{MI_NOMBRE}"

path_volume = f"/Volumes/{catalog}/{schema}/{volume}"

display(dbutils.fs.ls(path_volume))

In [0]:
file_path = f"{path_volume}/transactions_data.csv"

df = (
    spark.read.format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load(file_path)
)

print(f"Total de registros: {df.count():,}")
print(f"Columnas: {len(df.columns)}")
df.printSchema()
df.show(5, truncate=False)

In [0]:
df_sin_infer_schema = (
    spark.read.format("csv")
    .option("header", "true")
    .option("inferSchema", "false")
    .load(file_path)
)

df_sin_infer_schema.printSchema()
df_sin_infer_schema.show(5, truncate=False)

## Diferencia entre inferSchema=True e inferSchema=False

Cuando se usa `inferSchema=True`, Spark intenta detectar automáticamente los tipos de datos de cada columna. En este caso identificó columnas como `id`, `client_id`, `card_id`, `merchant_id` y `mcc` como enteros, y `date` como timestamp.

Cuando se usa `inferSchema=False`, Spark lee todas las columnas como string. Esto puede ser más rápido al momento de leer inicialmente, pero obliga a convertir manualmente columnas numéricas o de fecha antes de hacer análisis.

En este dataset, `amount` fue leído como string incluso con `inferSchema=True`, porque los valores vienen con símbolo de dinero y posiblemente signos negativos, por ejemplo valores tipo `$123.45` o `-$50.00`.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *

for field in df.schema.fields:
    print(f"{field.name}: {field.dataType}")

## Observaciones del schema

### Columnas que deberían ser numéricas pero Spark leyó como string

La columna `amount` debería ser numérica, pero Spark la leyó como string porque contiene símbolos como `$`, separadores o valores negativos.

La columna `zip` fue leída como double, aunque por lógica de negocio un código postal podría tratarse como texto, ya que no se usa para cálculos matemáticos.

### Columnas de fecha

La columna `date` fue leída como timestamp. El formato identificado corresponde a fecha y hora.

### Columnas categóricas

Las columnas que parecen categóricas son:

- `use_chip`
- `merchant_city`
- `merchant_state`
- `mcc`
- `errors`

También `client_id`, `card_id` y `merchant_id` son identificadores que pueden usarse para relaciones o agrupaciones, aunque técnicamente Spark los leyó como enteros.

In [0]:
df_sel = df.select(
    "id",
    "date",
    "client_id",
    "card_id",
    "amount",
    "use_chip",
    "merchant_city",
    "merchant_state",
    "mcc"
)

df_sel = (
    df_sel
    .withColumnRenamed("id", "transaction_id")
    .withColumnRenamed("date", "transaction_date")
    .withColumnRenamed("use_chip", "transaction_type")
)

df_sel.show(5, truncate=False)

## Diferencia entre select() y drop()

`select()` se usa cuando quiero escoger explícitamente las columnas que necesito conservar. Es útil cuando solo quiero trabajar con un subconjunto pequeño de columnas.

`drop()` se usa cuando quiero conservar casi todas las columnas y eliminar solo algunas específicas.

En este caso usé `select()` porque la actividad pide trabajar solo con columnas relevantes para el análisis inicial de transacciones.

In [0]:
df_typed = (
    df_sel
    .withColumn("amount", F.regexp_replace(F.col("amount"), "[$,]", "").cast("double"))
    .withColumn("transaction_date", F.to_timestamp(F.col("transaction_date"), "yyyy-MM-dd HH:mm:ss"))
)

df_typed.printSchema()
df_typed.select("amount", "transaction_date").show(10, truncate=False)

## Transformación de tipos

La columna `amount` venía como string porque incluía símbolo de dinero `$` y también puede contener valores negativos. Para poder hacer cálculos, eliminé los símbolos `$` y `,` usando `regexp_replace`, y luego convertí la columna a tipo `double`.

La columna `transaction_date` fue convertida a timestamp usando el formato `yyyy-MM-dd HH:mm:ss`.

Los valores negativos en `amount` tienen sentido en el negocio porque pueden representar devoluciones, retiros, reversos o ajustes sobre transacciones.

In [0]:
# Transacciones con monto mayor a $1,000
df_grandes = df_typed.filter(F.col("amount") > 1000)
print(f"Transacciones > $1,000: {df_grandes.count():,}")

# Transacciones con chip / swipe
df_chip = df_typed.filter(F.col("transaction_type") == "Swipe Transaction")
print(f"Transacciones con chip o swipe: {df_chip.count():,}")

# Transacciones negativas
df_negativos = df_typed.filter(F.col("amount") < 0)
print(f"Transacciones negativas: {df_negativos.count():,}")

display(df_grandes.limit(5))
display(df_chip.limit(5))
display(df_negativos.limit(5))

## Filtros y condiciones

### Transacciones mayores a $1,000

Se encontraron **X** transacciones con monto mayor a $1,000.  
Estas transacciones pueden ser relevantes para análisis de riesgo, monitoreo de operaciones grandes o segmentación de clientes de alto valor.

### Transacciones tipo Swipe Transaction

Se encontraron **Y** transacciones tipo `Swipe Transaction`.  
Este tipo de transacción representa operaciones presenciales realizadas con tarjeta física. Tiene sentido analizarlas por separado frente a transacciones online o con chip.

### Transacciones negativas

Se encontraron **Z** transacciones negativas.  
Estos valores tienen sentido en un contexto financiero porque pueden representar devoluciones, reversos, retiros, cargos corregidos o ajustes contables.

In [0]:
df_enriched = (
    df_typed
    .withColumn("year", F.year(F.col("transaction_date")))
    .withColumn("month", F.month(F.col("transaction_date")))
    .withColumn("day_of_week", F.dayofweek(F.col("transaction_date")))
    .withColumn("is_weekend", F.when(F.col("day_of_week").isin([1, 7]), True).otherwise(False))
    .withColumn("amount_abs", F.abs(F.col("amount")))
)

df_enriched.select(
    "transaction_id",
    "transaction_date",
    "year",
    "month",
    "day_of_week",
    "is_weekend",
    "amount",
    "amount_abs"
).show(10, truncate=False)

In [0]:
df_enriched.select(
    "transaction_date",
    "day_of_week",
    "is_weekend"
).orderBy("transaction_date").show(20, truncate=False)

## Nuevas columnas calculadas

Se agregaron columnas derivadas a partir de la fecha de la transacción:

- `year`: año de la transacción.
- `month`: mes de la transacción.
- `day_of_week`: día de la semana.
- `is_weekend`: indica si la transacción ocurrió en fin de semana.
- `amount_abs`: valor absoluto del monto, útil para analizar volumen sin importar si el movimiento fue positivo o negativo.

En PySpark, `dayofweek()` retorna:

- `1` para domingo.
- `2` para lunes.
- `7` para sábado.

Por eso se marcó como fin de semana cuando `day_of_week` es `1` o `7`.

In [0]:
# Transacciones por año
df_year_stats = (
    df_enriched.groupBy("year")
    .agg(
        F.count("transaction_id").alias("total_transacciones"),
        F.sum("amount_abs").alias("volumen_total"),
        F.avg("amount_abs").alias("ticket_promedio"),
        F.max("amount_abs").alias("monto_maximo")
    )
    .orderBy("year")
)

display(df_year_stats)

In [0]:
# Top 10 ciudades con más transacciones
df_top_cities = (
    df_enriched.groupBy("merchant_city")
    .count()
    .orderBy(F.col("count").desc())
    .limit(10)
)

display(df_top_cities)

In [0]:
# Distribución por tipo de transacción
df_transaction_type = (
    df_enriched.groupBy("transaction_type")
    .agg(
        F.count("*").alias("total"),
        F.avg("amount_abs").alias("ticket_promedio")
    )
    .orderBy(F.col("total").desc())
)

display(df_transaction_type)

## Agrupaciones y estadísticas

En esta sección se calcularon estadísticas agregadas para entender mejor el comportamiento del dataset.

Primero se agruparon las transacciones por año para revisar el volumen total, el ticket promedio y el monto máximo por periodo.

Luego se identificaron las 10 ciudades con mayor cantidad de transacciones. Este análisis permite detectar ciudades con mayor actividad financiera.

Finalmente, se revisó la distribución por tipo de transacción para comparar cuántas operaciones corresponden a cada canal y cuál es el ticket promedio por tipo.

In [0]:
from pyspark.sql.functions import col, sum as spark_sum, isnan, when

numeric_types = {"double", "float"}

nulos = df_enriched.select([
    spark_sum(
        when(
            col(c).isNull() | (isnan(col(c)) if df_enriched.schema[c].dataType.typeName() in numeric_types else False),
            1
        ).otherwise(0)
    ).alias(c)
    for c in df_enriched.columns
])

nulos.show(vertical=True)

In [0]:
nulos_dict = nulos.collect()[0].asDict()

columna_mas_nulos = max(nulos_dict, key=nulos_dict.get)
cantidad_mas_nulos = nulos_dict[columna_mas_nulos]

print(f"Columna con más nulos: {columna_mas_nulos}")
print(f"Cantidad de nulos: {cantidad_mas_nulos:,}")

## Análisis de valores nulos

La columna con más valores nulos fue `merchant_state`, con **1,563,700** registros nulos.

Esto puede afectar el análisis porque `merchant_state` representa el estado donde ocurrió la transacción. Si esta información falta, se dificulta hacer análisis geográfico por estado o detectar patrones regionales.

Sin embargo, no eliminaría automáticamente esas filas, porque la transacción sigue teniendo información útil como monto, fecha, cliente, tarjeta, ciudad, tipo de transacción y MCC.

Para un pipeline productivo, una opción sería conservar los registros y reemplazar `merchant_state` por una categoría como `unknown` cuando el análisis requiera segmentación geográfica. También revisaría si los nulos corresponden a transacciones online, internacionales o comercios donde el estado no aplica.

## Reflexión final

### PySpark vs pandas

Lo más difícil de entender al pasar de pandas a PySpark fue que en Spark muchas operaciones no se ejecutan inmediatamente. En pandas normalmente una transformación se aplica directamente sobre el DataFrame en memoria, mientras que en Spark se construye un plan de ejecución que solo corre cuando se ejecuta una acción como `count()`, `show()` o `display()`.

Una ventaja importante de Spark en Databricks es que permite trabajar con datasets grandes sin cargar todo en memoria local. En este caso, el archivo `transactions_data.csv` tiene más de 13 millones de registros, lo cual sería más pesado de manejar directamente con pandas en un equipo local.

Las celdas que ejecutan acciones como `count()`, `show()` o agrupaciones tardaron más porque Spark realmente tuvo que leer, procesar o agregar datos. Las celdas que solo definen transformaciones fueron más rápidas porque Spark todavía no ejecutaba el plan completo.

### Sobre el dataset

Un hallazgo interesante fue que la columna `amount` llegó como string, aunque representa valores numéricos. Esto ocurrió porque contiene símbolos como `$` y también valores negativos. Para poder hacer análisis fue necesario limpiarla y convertirla a tipo `double`.

También llamó la atención que la columna con más nulos fue `merchant_state`, con **1,563,700** registros nulos. Esto puede afectar análisis geográficos por estado, pero no necesariamente invalida las transacciones.

Con esta tabla se podrían responder preguntas de negocio como:

- ¿En qué años hubo mayor volumen de transacciones?
- ¿Cuáles son las ciudades con más actividad financiera?
- ¿Qué tipo de transacción tiene mayor ticket promedio?
- ¿Cuántas transacciones son negativas?
- ¿Qué patrones existen entre transacciones presenciales y online?

### Evaluación lazy

Sí se nota la diferencia entre encadenar transformaciones y ejecutar una acción. Cuando se usa `select()`, `withColumn()` o `filter()`, Spark solo construye el plan de ejecución. El código realmente se ejecuta cuando se llama una acción como `count()`, `show()`, `display()` o `write()`.

Esto es importante porque permite optimizar el plan antes de ejecutarlo, pero también puede confundir al inicio porque una celda puede parecer rápida aunque todavía no haya procesado los datos realmente.

## Parte extra — Internals de Spark

En esta sección se valida cómo Spark ejecuta realmente las transformaciones.  
El objetivo es diferenciar entre transformaciones lazy y acciones, además de revisar conceptos como driver, executors, jobs, stages y tasks.

In [0]:
import time
from pyspark.sql.functions import col, sum as spark_sum, regexp_replace

# Crear una versión con amount_num para la demostración
df_lazy = df.withColumn(
    "amount_num",
    regexp_replace(col("amount"), r"[$,]", "").cast("double")
)

# Construir el plan — esto NO ejecuta el procesamiento completo
t0 = time.time()

df_plan = (
    df_lazy
    .filter(col("amount_num") > 0)
    .groupBy("merchant_city")
    .agg(spark_sum("amount_num").alias("total"))
)

t1 = time.time()
tiempo_construccion = t1 - t0

print(f"Construir transformaciones lazy: {tiempo_construccion:.6f} segundos")

In [0]:
t0 = time.time()

df_plan.show(20, truncate=False)

t1 = time.time()
tiempo_ejecucion_show = t1 - t0

print(f"Ejecutar con show(): {tiempo_ejecucion_show:.4f} segundos")

In [0]:
df_paso1 = df_lazy.filter(col("amount_num") > 0)
df_paso2 = df_paso1.withColumn("amount_abs", F.abs(col("amount_num")))
df_paso3 = df_paso2.groupBy("merchant_city").count()

print("Se crearon tres transformaciones encadenadas.")
print("Hasta este punto Spark todavía no ejecuta el procesamiento completo.")

In [0]:
t0 = time.time()

n = df_paso3.count()

t1 = time.time()
tiempo_count = t1 - t0

print(f"Ciudades de comercio: {n}")
print(f"Tiempo ejecutando count(): {tiempo_count:.4f} segundos")

## Evaluación lazy

Al construir el DataFrame `df_plan`, Spark tardó muy poco tiempo porque solo estaba creando el plan lógico de ejecución. Esa operación fue lazy y no procesó todo el dataset todavía.

La ejecución real ocurrió cuando se llamó una acción como `show()` o `count()`.

Tiempo construyendo transformaciones: **X segundos**  
Tiempo ejecutando `show()`: **Y segundos**  
Tiempo ejecutando `count()`: **Z segundos**

Llegué a encadenar tres transformaciones (`filter`, `withColumn`, `groupBy`) sin que Spark ejecutara el procesamiento completo. Spark ejecutó realmente el plan cuando se llamó `count()`.

In [0]:
t0 = time.time()

display(df_plan.limit(10))

t1 = time.time()
tiempo_display = t1 - t0

print(f"Ejecutar con display(limit(10)): {tiempo_display:.4f} segundos")

## show() vs display()

`show()` es una acción estándar de PySpark. Funciona en cualquier entorno Spark y muestra los resultados como texto plano en la salida de la celda. Es útil para depuración rápida o pipelines donde no hay interfaz gráfica.

`display()` es una función propia de Databricks. También dispara una acción, pero muestra el resultado como una tabla interactiva, con opciones visuales y exploración más cómoda.

Preferiría `show()` cuando esté trabajando en scripts, jobs automatizados, pruebas o ambientes sin interfaz visual. Preferiría `display()` cuando esté explorando datos dentro de Databricks porque permite revisar los resultados de forma más clara.

Tiempo ejecutando `display(limit(10))`: 6.5625 segundos

## Driver y Executors

En Spark, el Driver es el proceso principal que interpreta el código, construye el plan de ejecución y coordina el trabajo.

Los Executors son los procesos que ejecutan las tareas sobre las particiones de datos.

En Databricks Serverless no siempre se puede acceder directamente a `spark.sparkContext`, porque el entorno abstrae parte del driver. Por eso se usan alternativas como `spark.version` y `spark.conf.get()`.

In [0]:
print(f"Versión Spark: {spark.version}")
print(f"Particiones SQL actuales: {spark.conf.get('spark.sql.shuffle.partitions')}")

In [0]:
spark.conf.set("spark.sql.shuffle.partitions", "16")

print("Particiones ajustadas a 16")
print(f"Particiones SQL nuevas: {spark.conf.get('spark.sql.shuffle.partitions')}")

## Configuración de particiones

El valor `spark.sql.shuffle.partitions` define cuántas particiones usa Spark en operaciones que requieren shuffle, como `groupBy`, `join` u `orderBy`.

El valor por defecto suele ser 200. Para datasets medianos y entornos pequeños, ese número puede ser excesivo y generar demasiadas tareas pequeñas.

En este notebook ajusté el valor a `16` para trabajar de forma más razonable en un entorno pequeño de Databricks.

In [0]:
resultado = (
    df_lazy
    .groupBy("merchant_city")
    .agg(spark_sum("amount_num").alias("total"))
    .orderBy(col("total").desc())
    .limit(10)
)

resultado.show(truncate=False)

## Job, Stage y Task — Spark UI

Al ejecutar `resultado.show()`, Spark creó un Job porque `show()` es una acción.

Desde `See performance` pude confirmar que la celda generó una ejecución asociada al `show()`. Además, al revisar el plan con `resultado.explain(True)`, se observa la operación `Exchange`, lo que indica que Spark realizó shuffle.

Las operaciones `groupBy` y `orderBy` generan reorganización de datos entre particiones, por eso Spark divide la ejecución en stages.

No pude validar con precisión el número exacto de stages y tasks desde la interfaz de Databricks Serverless, pero sí confirmé que la acción `show()` dispara la ejecución real del plan y que el shuffle aparece en el plan físico como `Exchange`.

El análisis del Spark UI permite entender dónde se consume más tiempo y qué operaciones generan shuffle.

In [0]:
print(type(spark))
print(f"Spark version: {spark.version}")

## SparkSession

En Databricks, la variable `spark` ya está disponible automáticamente cuando se abre un notebook. Esta variable representa una `SparkSession`, que es el punto de entrada para trabajar con DataFrames, SQL y configuraciones de Spark.

En otros entornos, como una ejecución local o un pipeline fuera de Databricks, normalmente se debe crear manualmente con `SparkSession.builder`.

En Databricks Serverless puede fallar el acceso directo a `spark.sparkContext`, por eso en este notebook se usaron alternativas como `spark.version` y `spark.conf.get()`.

In [0]:
print(type(spark))
print(f"Spark version: {spark.version}")

## SparkSession

En Databricks, la variable `spark` ya está disponible automáticamente cuando se abre un notebook. Esta variable representa una `SparkSession`, que es el punto de entrada para trabajar con DataFrames, SQL y configuraciones de Spark.

En otros entornos, como una ejecución local o un pipeline fuera de Databricks, normalmente se debe crear manualmente con `SparkSession.builder`.

En Databricks Serverless puede fallar el acceso directo a `spark.sparkContext`, por eso en este notebook se usaron alternativas como `spark.version` y `spark.conf.get()`.